## Setup: Import Required Functions

In [3]:
from pathlib import Path
import sys
# set current cwd parent to path
sys.path.append(str(Path.cwd().parent.parent.resolve()))

from utils4e import *
from logic4e import *

# Define symbols for examples
P, Q, A, B, C, D, E, F, G = symbols('P, Q, A, B, C, D, E, F, G')

## The Trace Decorator

This decorator wraps `tt_check_all()` to show:
1. **Recursion structure** - Call number and depth
2. **Model building** - Truth assignments growing from {} to complete
3. **Symbol processing** - Which symbol at each level
4. **Binary branches** - Symbol = True and Symbol = False paths
5. **Evaluation** - KB and query evaluation at leaf nodes
6. **Combination** - AND-ing results from both branches

In [7]:
# decorator function

def trace_tt_check_all(func):
    """
    Decorator that traces tt_check_all execution with detailed output.
    Shows recursion depth, model building, symbol processing, and evaluation.
    """
    call_depth = [0]  # Use list to allow modification in nested function
    
    def wrapper(kb, alpha, symbols, model):
        
        # increment call stack at each call
        call_depth[0] += 1

        indent = "  " * (call_depth[0] - 1)
        call_id = call_depth[0]
        
        print(f"\n{'='*80}")
        print(f"{indent}[CALL #{call_id}] tt_check_all()")
        print(f"{indent}├─ Recursion Depth: {call_depth[0]}")
        print(f"{indent}├─ Model: {model}")
        print(f"{indent}├─ Remaining Symbols: {symbols}")
        print(f"{indent}├─ KB Expression: {kb}")
        print(f"{indent}└─ Alpha Query: {alpha}")
        
        # Base case: no more symbols
        if not symbols:
            print(f"{indent}\n[BASE CASE] No more symbols to assign")
            print(f"{indent}├─ Complete Model: {model}")
            
            # Evaluate KB with complete model
            kb_result = pl_true(kb, model)
            print(f"{indent}├─ KB Evaluation: pl_true({kb}, model) = {kb_result}")
            
            if kb_result:
                # Evaluate alpha with complete model
                alpha_result = pl_true(alpha, model)
                print(f"{indent}├─ KB is TRUE → Evaluate Alpha")
                print(f"{indent}├─ Alpha Evaluation: pl_true({alpha}, model) = {alpha_result}")
                print(f"{indent}└─ RETURN: {alpha_result}")
                result = alpha_result
            else:
                print(f"{indent}├─ KB is FALSE → Implication is vacuously TRUE")
                print(f"{indent}└─ RETURN: True")
                result = True
        else:
            # Recursive case
            P = symbols[0]
            rest = symbols[1:]
            
            print(f"{indent}\n[RECURSIVE CASE] Processing symbol: {P}")
            print(f"{indent}├─ Next Symbol to Branch: {P}")
            print(f"{indent}├─ Remaining Symbols after {P}: {rest}")
            
            # Branch 1: Set P to True
            print(f"{indent}\n{indent}[BRANCH 1] Setting {P} = True")
            model_true = extend(model, P, True)
            print(f"{indent}├─ Extended Model: {model_true}")

            # recursive call Branch 1 - True
            result_true = wrapper(kb, alpha, rest, model_true)
            print(f"{indent}├─ Branch 1 Result: {result_true}")
            
            # Branch 2: Set P to False
            print(f"{indent}\n{indent}[BRANCH 2] Setting {P} = False")
            model_false = extend(model, P, False)
            print(f"{indent}├─ Extended Model: {model_false}")

            # recursive call Branch 2 - False
            result_false = wrapper(kb, alpha, rest, model_false)
            print(f"{indent}├─ Branch 2 Result: {result_false}")
            
            # Combine results with AND
            result = result_true and result_false
            print(f"{indent}\n{indent}[COMBINE] Both branches must be TRUE")
            print(f"{indent}├─ {result_true} AND {result_false} = {result}")
            print(f"{indent}└─ RETURN: {result}")
        
        call_depth[0] -= 1
        return result
    
    return wrapper


# Create the traced version
traced_tt_check_all = trace_tt_check_all(tt_check_all)

def tt_entails_traced(kb, alpha):
    """Wrapper for tt_entails that uses the traced version."""
    assert not variables(alpha)
    symbols = list(prop_symbols(kb & alpha))
    print(f"\n{'='*80}")
    print(f"[STARTING tt_entails_traced]")
    print(f"├─ KB: {kb}")
    print(f"├─ Alpha (Query): {alpha}")
    print(f"├─ All Symbols to Process: {symbols}")
    print(f"└─ Initial Model: {{}}\n")
    
    result = traced_tt_check_all(kb, alpha, symbols, {})
    
    print(f"\n{'='*80}")
    print(f"[FINAL RESULT] tt_entails({kb}, {alpha}) = {result}")
    print(f"{'='*80}\n")
    return result

## Example 1: Successful Entailment

**Question:** Does (P ∧ Q) entail Q?

**Expected Answer:** YES (True)

**Explanation:** The only model satisfying (P ∧ Q) is {P: True, Q: True}. In this model, Q is also true.

In [8]:
result1 = tt_entails_traced(P & Q, Q)


[STARTING tt_entails_traced]
├─ KB: (P & Q)
├─ Alpha (Query): Q
├─ All Symbols to Process: [Q, P]
└─ Initial Model: {}


[CALL #1] tt_check_all()
├─ Recursion Depth: 1
├─ Model: {}
├─ Remaining Symbols: [Q, P]
├─ KB Expression: (P & Q)
└─ Alpha Query: Q

[RECURSIVE CASE] Processing symbol: Q
├─ Next Symbol to Branch: Q
├─ Remaining Symbols after Q: [P]

[BRANCH 1] Setting Q = True
├─ Extended Model: {Q: True}

  [CALL #2] tt_check_all()
  ├─ Recursion Depth: 2
  ├─ Model: {Q: True}
  ├─ Remaining Symbols: [P]
  ├─ KB Expression: (P & Q)
  └─ Alpha Query: Q
  
[RECURSIVE CASE] Processing symbol: P
  ├─ Next Symbol to Branch: P
  ├─ Remaining Symbols after P: []
  
  [BRANCH 1] Setting P = True
  ├─ Extended Model: {Q: True, P: True}

    [CALL #3] tt_check_all()
    ├─ Recursion Depth: 3
    ├─ Model: {Q: True, P: True}
    ├─ Remaining Symbols: []
    ├─ KB Expression: (P & Q)
    └─ Alpha Query: Q
    
[BASE CASE] No more symbols to assign
    ├─ Complete Model: {Q: True, P: True}
  

## Example 2: Failed Entailment

**Question:** Does (P ∨ Q) entail Q?

**Expected Answer:** NO (False)

**Explanation:** (P ∨ Q) is true in three models: {P: T, Q: F}, {P: T, Q: T}, {P: F, Q: T}. In the first model, Q is false. So entailment fails.

In [9]:
result2 = tt_entails_traced(P | Q, Q)


[STARTING tt_entails_traced]
├─ KB: (P | Q)
├─ Alpha (Query): Q
├─ All Symbols to Process: [Q, P]
└─ Initial Model: {}


[CALL #1] tt_check_all()
├─ Recursion Depth: 1
├─ Model: {}
├─ Remaining Symbols: [Q, P]
├─ KB Expression: (P | Q)
└─ Alpha Query: Q

[RECURSIVE CASE] Processing symbol: Q
├─ Next Symbol to Branch: Q
├─ Remaining Symbols after Q: [P]

[BRANCH 1] Setting Q = True
├─ Extended Model: {Q: True}

  [CALL #2] tt_check_all()
  ├─ Recursion Depth: 2
  ├─ Model: {Q: True}
  ├─ Remaining Symbols: [P]
  ├─ KB Expression: (P | Q)
  └─ Alpha Query: Q
  
[RECURSIVE CASE] Processing symbol: P
  ├─ Next Symbol to Branch: P
  ├─ Remaining Symbols after P: []
  
  [BRANCH 1] Setting P = True
  ├─ Extended Model: {Q: True, P: True}

    [CALL #3] tt_check_all()
    ├─ Recursion Depth: 3
    ├─ Model: {Q: True, P: True}
    ├─ Remaining Symbols: []
    ├─ KB Expression: (P | Q)
    └─ Alpha Query: Q
    
[BASE CASE] No more symbols to assign
    ├─ Complete Model: {Q: True, P: True}
  

## Understanding the Algorithm

### What is tt_check_all()?

It's a **truth table enumeration** algorithm that checks if KB ⊨ α:

1. **Enumerate all 2^n models** (truth assignments for n symbols)
2. **For each model:**
   - If KB is true in the model, check if α is also true
   - If KB is false, skip (vacuous truth)
3. **Entailment holds** if α is true in all models where KB is true

### Key Mathematical Principle

$$\text{KB} \vDash \alpha \iff \text{KB} \land \neg\alpha \text{ is unsatisfiable}$$

The algorithm checks this by finding models that satisfy both KB and ¬α. If none exist, entailment holds.

### Recursion Structure

For symbols [P, Q, R]:

```
tt_check_all([P,Q,R], {})         ← depth 1
├─ P=T → tt_check_all([Q,R], {P:T})    ← depth 2
│        ├─ Q=T → tt_check_all([R], {P:T,Q:T})  ← depth 3
│        │        ├─ R=T → [BASE CASE] {P:T,Q:T,R:T}
│        │        └─ R=F → [BASE CASE] {P:T,Q:T,R:F}
│        └─ Q=F → tt_check_all([R], {P:T,Q:F})  ← depth 3
│                 ├─ R=T → [BASE CASE] {P:T,Q:F,R:T}
│                 └─ R=F → [BASE CASE] {P:T,Q:F,R:F}
└─ P=F → tt_check_all([Q,R], {P:F})    ← depth 2
         ├─ Q=T → tt_check_all([R], {P:F,Q:T})  ← depth 3
         │        ├─ R=T → [BASE CASE] {P:F,Q:T,R:T}
         │        └─ R=F → [BASE CASE] {P:F,Q:T,R:F}
         └─ Q=F → tt_check_all([R], {P:F,Q:F})  ← depth 3
                  ├─ R=T → [BASE CASE] {P:F,Q:F,R:T}
                  └─ R=F → [BASE CASE] {P:F,Q:F,R:F}

Total leaf nodes (base cases) = 2^3 = 8 complete models
```

## Reading the Trace Output

### Output Symbols

| Symbol | Meaning |
|--------|----------|
| `[CALL #n]` | New recursive call at depth n |
| `[BASE CASE]` | Leaf node - complete model evaluated |
| `[RECURSIVE CASE]` | Internal node - symbols remaining |
| `[BRANCH 1]` | Explore symbol = True |
| `[BRANCH 2]` | Explore symbol = False |
| `[COMBINE]` | AND results from both branches |
| `[FINAL RESULT]` | Overall entailment verdict |

### Indentation

- **No indent** = Depth 1 (root call)
- **2 spaces** = Depth 2 (first symbol assigned)
- **4 spaces** = Depth 3 (second symbol assigned)
- **6 spaces** = Depth 4, etc.

Deeper indentation = deeper in recursion tree = more symbols assigned.

### Base Case Evaluation

At each base case (leaf node):
1. **Evaluate KB** in complete model
2. If KB = TRUE:
   - **Evaluate α** in same model
   - Return α's truth value
3. If KB = FALSE:
   - **Return True** (vacuous truth: false → anything is true)

### Combining Results

Both branches must return True for the parent to return True:
```
[COMBINE] result_true AND result_false = final_result
```

This ensures entailment only if α is true in **ALL** models where KB is true.

## Computational Complexity

### Time Complexity: O(2^n)

| # Symbols | # Models | Status |
|-----------|----------|--------|
| 1         | 2        | ✓ Instant |
| 3         | 8        | ✓ Instant |
| 5         | 32       | ✓ OK |
| 10        | 1,024    | ✓ Still manageable |
| 20        | 1,048,576 | ✗ Slow |
| 30        | ~1 billion | ✗ Very slow |

### Why This Matters

Truth table enumeration is **exponential** and doesn't scale:

- **DPLL** (Davis-Putnam): Uses heuristics (pure symbols, unit propagation) to prune search space
- **WalkSAT**: Uses randomized local search instead of exhaustive enumeration
- **Modern SAT Solvers**: Highly optimized with learned clauses, restarts, and smart heuristics

This trace shows **why** these alternatives are necessary for practical problems.

## Summary: What You've Learned

✓ **How truth table enumeration works** - exhaustive model checking  
✓ **Recursion structure** - binary tree with depth = number of symbols  
✓ **Model building** - progressive assignment from {} to complete  
✓ **Evaluation** - checking KB and α only at leaf nodes  
✓ **Entailment** - KB ⊨ α iff α true in all models where KB true  
✓ **Why it matters** - understanding why SAT solvers use heuristics  
✓ **Complexity** - exponential growth limits practical applicability  

### Key Takeaway

This tracer reveals the **complete execution** of a fundamental algorithm in logical inference. While truth table enumeration is elegant and theoretically important, its exponential nature necessitates the intelligent pruning strategies used in modern inference engines.

---

**Reference:** Chapter 7 (Logical Agents), AIMA 4th Edition